# 01 - Setup and Data Loading

Configuration, dependency installation, GloVe loading, and raw city file loading.

> Split from `PersonaMatch.ipynb` without re-running cells; saved outputs are preserved as stored in the original notebook.


**Data Loading & Configuration:**

In [0]:
# NOTE:
# SAS token is NOT included in the repository.
# Users must provide their own credentials to access the data.
# config
storage_account = "lab94290"
container = "airbnb"
user_email = "edaniel@campus.technion.ac.il"
base_path = f"file:/Workspace/Users/{user_email}/"

# SAS Token configuration
sas_token = "Put the SAS token here"
sas_token = sas_token.lstrip('?')

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "SAS")
spark.conf.set(f"fs.azure.sas.token.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider")
spark.conf.set(f"fs.azure.sas.fixed.token.{storage_account}.dfs.core.windows.net", sas_token)

path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/airbnb_1_12_parquet"
airbnb_df = spark.read.parquet(path)

In [0]:
%pip install gensim

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Loading and Broadcasting Pre-trained Word Embeddings (GloVe)**

In [0]:
import gensim.downloader as api
import numpy as np

# Downloading the model from Gensim servers
print("Downloading Word2Vec model...")
w2v_model = api.load("glove-wiki-gigaword-100") 
print("Model loaded successfully!")

# Broadcast the model to all workers in the cluster
bc_w2v_model = spark.sparkContext.broadcast(w2v_model)

Model loaded successfully!


In [0]:
# Loading all files from the list in the image

# List of all cities that appear in your Workspace
cities = [
    "barcelona", "budapest", "buenos_aires", "cape_town", 
    "lisbon", "mexico_city", "miami", "new_york", 
    "paris", "phuket", "san_francisco", "sau_paulo", 
    "tel_aviv", "tiberias", "tokyo"
]

# A dictionary that will contain all the DataFrames (so you don't have to manually define a variable for each one)
dfs = {}

for city in cities:
    file_path = f"{base_path}{city}_final.csv"
    try:
        dfs[city] = (spark.read.format("csv")
                      .option("header", "true")
                      .option("inferSchema", "true")
                      .load(file_path))
        print(f"Successfully loaded: {city}")
    except Exception as e:
        print(f"Error loading {city}: {e}")

# Now you can access any DataFrame conveniently, for example: dfs['tel_aviv'].show()
print(f"\nTotal cities loaded: {len(dfs)}")

Successfully loaded: barcelona
Successfully loaded: budapest
Successfully loaded: buenos_aires
Successfully loaded: cape_town
Successfully loaded: lisbon
Successfully loaded: mexico_city
Successfully loaded: miami
Successfully loaded: new_york
Successfully loaded: paris
Successfully loaded: phuket
Successfully loaded: san_francisco
Successfully loaded: sau_paulo
Successfully loaded: tel_aviv
Successfully loaded: tiberias
Successfully loaded: tokyo

Total cities loaded: 15
